# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

# Reuse the project's own feature lists + precision@K definition (scripts/ml_utils.py)
# instead of redefining them here, so Section 2/3 stay consistent with the Week-4 baseline.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "scripts" / "ml_utils.py").exists():
    REPO_ROOT = REPO_ROOT.parent
SCRIPTS_DIR = REPO_ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, PROCESSED_DIR, precision_at_k

FEATURE_PATH = PROCESSED_DIR / "refresh_feature_vector.csv"
BASELINE_PATH = PROCESSED_DIR / "baseline_refresh_queue.csv"

# Build the Week-4 baseline artifacts if this is a fresh checkout / fresh runtime.
if not FEATURE_PATH.exists():
    subprocess.run([sys.executable, "01_prepare_features.py"], cwd=SCRIPTS_DIR, check=True)
if not BASELINE_PATH.exists():
    subprocess.run([sys.executable, "02_baseline_score.py"], cwd=SCRIPTS_DIR, check=True)

df = pd.read_csv(FEATURE_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)

# ---- Build the model's feature matrix (same feature lists as scripts/03_train_model.py) ----
numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

numeric_frame = (
    df[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)
categorical_frame = pd.get_dummies(
    df[categorical_features].fillna("unknown").astype(str),
    prefix=categorical_features,
    dtype=float,
)
X = pd.concat([numeric_frame.reset_index(drop=True), categorical_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int).reset_index(drop=True)
groups = df["client_id"].astype(str).reset_index(drop=True)

# ---- The Comparison Contract: baseline score aligned row-for-row with the model's frame ----
# baseline_refresh_score was computed once on the Week-4 data, keyed by content_id. We map it
# onto this exact row order so every fold's rule score and model score share identical rows.
baseline_lookup = baseline_df.set_index("content_id")["baseline_refresh_score"]
baseline_score_full = df["content_id"].map(baseline_lookup).fillna(0).reset_index(drop=True)

assert len(X) == len(y) == len(groups) == len(baseline_score_full), "Row counts must match before splitting"

# ---- Split design: GroupKFold grouped by client_id ----
# Random row splits let the model memorize a client's quirks and fake skill. Grouping by
# client_id means every client's rows land entirely in train OR entirely in validation for
# each fold, so the honest question gets asked: does this generalize to a client never seen?
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

print(f"Rows: {len(df):,} | Unique clients: {groups.nunique()} | Base rate (is_declining_label): {y.mean():.3f}\n")
print("Fold diagnostics (proves the split is honest -- no client appears on both sides):")
for fold_id, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    train_clients = set(groups.iloc[train_idx])
    val_clients = set(groups.iloc[val_idx])
    overlap = train_clients & val_clients
    print(
        f"  Fold {fold_id}: train={len(train_idx):>5} rows / {len(train_clients):>2} clients | "
        f"val={len(val_idx):>5} rows / {len(val_clients):>2} clients | "
        f"val base rate={y.iloc[val_idx].mean():.3f} | client overlap={len(overlap)}"
    )
    assert len(overlap) == 0, f"Fold {fold_id} leaks client(s) across train/val: {overlap}"

print("\nSplit is client-grouped and leakage-free: every client_id lives on exactly one side of each fold.")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# ---- Section 3: Train Logistic Regression under the same GroupKFold, score the baseline on
# the same folds -- the Comparison Contract: rule and model see identical rows and labels. ----
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
K = 50

oof_model_scores = np.zeros(len(y))
fold_rows = []

for fold_id, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ])
    model.fit(X_train, y_train)

    val_probabilities = model.predict_proba(X_val)[:, 1]
    oof_model_scores[val_idx] = val_probabilities

    # Same val_idx rows, same y_val labels, same K -- for BOTH the rule and the model.
    val_baseline_scores = baseline_score_full.to_numpy()[val_idx]

    fold_rows.append({
        "fold": fold_id,
        "val_rows": len(val_idx),
        "base_rate": float(y_val.mean()),
        "baseline_precision_at_50": precision_at_k(y_val, val_baseline_scores, K),
        "logreg_precision_at_50": precision_at_k(y_val, val_probabilities, K),
    })

fold_table = pd.DataFrame(fold_rows)
print("Per-fold comparison -- same held-out rows/labels feed the rule AND the model:\n")
print(fold_table.to_string(index=False))

# ---- Final comparison table: pooled out-of-fold precision@50 ----
# Every row's model score above is that row's OWN held-out prediction (never seen in training),
# so pooling across folds and scoring against the baseline on the same rows is still honest.
comparison_table = pd.DataFrame({
    "candidate": [
        "Base rate (random ranking)",
        "Week-4 rule baseline",
        "Logistic Regression (GroupKFold, out-of-fold)",
    ],
    "precision_at_50": [
        float(y.mean()),
        precision_at_k(y, baseline_score_full, K),
        precision_at_k(y, oof_model_scores, K),
    ],
})
print("\nFinal comparison table (out-of-fold precision@50, identical rows/labels for rule and model):\n")
print(comparison_table.to_string(index=False))

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.